In [ ]:
import numpy as np
from functools import partial
import uuid
import time
import pickle
import gzip

from tqdm.notebook import tqdm

import matplotlib.pyplot as plt

from qphaset.models import drmg_gstate_mps
from qphaset.annni import model_annni_ext

## Template notebook for running the DMRG for the ANNNI model

In [ ]:
def run_model(params, *, model_factory, gstate_solver):
    gstates, exec_times = [], []
    psi = None
    for (x, y) in tqdm(params):
        model = model_factory(x, y)
        timer = time.monotonic()
        # TODO Set the guess according to the grid, this is not 100% correct.
        psi = gstate_solver(model, guess_psi=psi)
        timer = time.monotonic() - timer
        gstates.append(psi)
        exec_times.append(timer)
    return gstates, {'times': exec_times}

def polar(r, phi):
    return r * np.array([np.cos(phi), np.sin(phi)])

In [ ]:
# *** Data sampling (Hamitonian parameters grid) ***

# n = 64  # Sampling grid size.
n = 64

# params = np.linspace(0.01, 1.5, n), np.linspace(0.01, 1.5, n)
# params = np.linspace(0.01, 1.5, n), np.linspace(1.5, 0.01, n)
# params = np.linspace(0.01, 1., n), np.linspace(1., 0.01, n)   # Good one
# params = np.linspace(0.5, 1.5, n), np.linspace(1., 0.2, n)      # Floating phase detail
# params = np.linspace(0.5, 1.5, n), np.linspace(0.8, 0.01, n)      # Floating phase detail (lowered)
# params = np.linspace(0.01, 1., n), np.linspace(0.8, 0.01, n)      # Floating phase detail (lowered) + multi-critical point
params = np.linspace(0.6, 0.8, n), np.linspace(0.4, 0.1, n)      # Floating phase detail ++

params = map(lambda m: m.flatten(), np.meshgrid(*params, indexing='xy'))
params = tuple(params)
params = np.stack(params).T

In [ ]:
# *** Config ***

l = 15  # Number of spins (l * 2 for ANNNI)

# model_factory = model_annni
model_name = 'annni_ext'
# model_factory = partial(model_annni_ext, c1=0)
model_factory = partial(model_annni_ext, c1=-1)
# model_factory = partial(model_annni_ext, c1=-0.1) # too low, but good for floating detail

drmg_params = {
    #'mixer': True,
    #'max_E_err': 1.e-16,
    #'chi_list': {0: 25, 10: 50, 20: 100},
    'trunc_params': {
       'chi_max': 64,
    #   'svd_min': 1.e-16
    },
    'max_hours': 16 / 3600
    #'combine': True
}

In [ ]:
# *** Solver ***

if model_name is None:
    model_name = model_factory.__name__
gstates, stats = run_model(params,
                           model_factory=partial(model_factory, l=l),
                           gstate_solver=partial(drmg_gstate_mps, drmg_params=drmg_params))

In [ ]:
# *** Execution time per pixel ***

plt.matshow(np.array(stats['times']).reshape((n, n)), cmap='hot', origin='lower')
plt.colorbar()

In [ ]:
# *** Data export ***

filename = f'{model_name}-{str(uuid.uuid4())}.pkl' 
data = dict(params=params, drmg_params=drmg_params,
            l=l, n=n,
            gstates=gstates, stats=stats)
with gzip.open(filename, 'wb') as f:
    pickle.dump(data, f)